# 04 · Aprendizaje no supervisado: clustering y reducción dimensional

Cuando no tenemos etiquetas, buscamos estructura: grupos, densidad, representaciones compactas o anomalías. Este lab amplía K-Means/PCA con clustering jerárquico, DBSCAN, Gaussian Mixtures, t-SNE y UMAP.

## Objetivos
- Entender qué significa 'cluster' según diferentes algoritmos.
- Seleccionar K con elbow y silhouette sin confundirlos con verdad de negocio.
- Comparar centroides, jerarquías, densidad y mezclas probabilísticas.
- Reducir dimensión para compresión y visualización.
- Entender por qué t-SNE/UMAP son herramientas de visualización, no sustitutos automáticos de PCA.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_moons, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, adjusted_rand_score
SEED=42
X,y=load_iris(return_X_y=True,as_frame=True); Xs=StandardScaler().fit_transform(X)

## 1. K-Means
Minimiza la suma de distancias cuadráticas dentro de clusters:
$$\sum_i ||x_i-\mu_{c_i}||^2$$

Supone implícitamente clusters aproximadamente esféricos y comparables en escala. Es sensible a outliers y a la inicialización, aunque `k-means++` ayuda.


In [ ]:
rows=[]
for k in range(2,9):
    km=KMeans(n_clusters=k,n_init=20,random_state=SEED).fit(Xs)
    rows.append([k,km.inertia_,silhouette_score(Xs,km.labels_)])
df=pd.DataFrame(rows,columns=['k','inertia','silhouette']); display(df.round(3))
fig,ax=plt.subplots(1,2,figsize=(10,4)); ax[0].plot(df.k,df.inertia,'o-'); ax[0].set_title('Elbow / inertia'); ax[1].plot(df.k,df.silhouette,'o-'); ax[1].set_title('Silhouette'); plt.show()

## 2. Clustering jerárquico
Construye una jerarquía de fusiones. `single`, `complete`, `average` y `ward` definen cómo medir distancia entre grupos. Es útil cuando queremos explorar estructuras anidadas o no conocemos K de antemano.


In [ ]:
for linkage in ['ward','complete','average','single']:
    labels=AgglomerativeClustering(n_clusters=3,linkage=linkage).fit_predict(Xs)
    print(linkage,'silhouette=',round(silhouette_score(Xs,labels),3),'ARI=',round(adjusted_rand_score(y,labels),3))

## 3. DBSCAN: clusters por densidad + ruido
DBSCAN no necesita K. Usa `eps` y `min_samples` para identificar regiones densas y marca outliers con etiqueta `-1`. Puede descubrir formas no convexas que K-Means no puede representar. Su debilidad: un único `eps` funciona mal si la densidad varía mucho. HDBSCAN extiende esta idea.


In [ ]:
Xm,ym=make_moons(n_samples=500,noise=.08,random_state=SEED); Xms=StandardScaler().fit_transform(Xm)
models={
 'KMeans':KMeans(n_clusters=2,n_init=20,random_state=SEED),
 'DBSCAN':DBSCAN(eps=.25,min_samples=8)
}
fig,ax=plt.subplots(1,2,figsize=(10,4))
for a,(name,m) in zip(ax,models.items()):
    lab=m.fit_predict(Xms); a.scatter(Xm[:,0],Xm[:,1],c=lab,s=15); a.set_title(name)
plt.show()

## 4. Gaussian Mixture Models (GMM)
K-Means asigna un punto a un solo cluster. GMM modela una mezcla de gaussianas y puede devolver **probabilidades de pertenencia**. Permite clusters elípticos mediante diferentes matrices de covarianza. La selección de componentes puede apoyarse en AIC/BIC.


In [ ]:
rows=[]
for k in range(1,8):
    gm=GaussianMixture(n_components=k,covariance_type='full',random_state=SEED).fit(Xs)
    rows.append([k,gm.aic(Xs),gm.bic(Xs)])
pd.DataFrame(rows,columns=['componentes','AIC','BIC'])

## 5. PCA: reducción lineal
PCA busca direcciones ortogonales de máxima varianza. Antes de PCA suele ser necesario estandarizar. A diferencia de t-SNE, PCA preserva una estructura global lineal y tiene transformación definida para nuevos datos.


In [ ]:
pca=PCA().fit(Xs)
print('varianza explicada acumulada:',np.cumsum(pca.explained_variance_ratio_))
plt.bar(range(1,len(pca.explained_variance_ratio_)+1),pca.explained_variance_ratio_); plt.plot(range(1,len(pca.explained_variance_ratio_)+1),np.cumsum(pca.explained_variance_ratio_),'o-'); plt.xlabel('componente'); plt.show()
Z=PCA(n_components=2).fit_transform(Xs); plt.scatter(Z[:,0],Z[:,1],c=y); plt.title('Iris en PCA 2D'); plt.show()

## 6. t-SNE y UMAP
t-SNE prioriza vecindades locales y es excelente para explorar embeddings o datos de alta dimensión, pero las distancias globales y tamaños de clusters pueden ser engañosos. UMAP suele ser más rápido, admite `transform` en algunas implementaciones y puede preservar mejor parte de la estructura global.


In [ ]:
ts=TSNE(n_components=2,perplexity=30,learning_rate='auto',init='pca',random_state=SEED).fit_transform(Xs)
plt.scatter(ts[:,0],ts[:,1],c=y); plt.title('t-SNE: usar para exploración visual'); plt.show()
# UMAP opcional en Colab:
# !pip -q install umap-learn
# import umap
# Z=umap.UMAP(n_neighbors=15,min_dist=.1,random_state=SEED).fit_transform(Xs)

## Aplicaciones
- segmentación de usuarios/hogares/territorios;
- exploración de embeddings de documentos;
- agrupación de imágenes;
- compresión antes de modelos;
- detección de subpoblaciones;
- exploración de cohorts y patrones operacionales.

## Errores comunes
- interpretar clusters como categorías 'reales' sin validación experta;
- elegir K solo por elbow;
- no escalar features;
- usar t-SNE como entrada directa a producción sin entender su naturaleza;
- comparar colores visualmente sin métricas;
- olvidar estabilidad de clusters ante cambios de muestra.

## Ejercicios
1. Compara K-Means, Agglomerative, DBSCAN y GMM en tres datasets sintéticos.
2. Usa `StandardScaler`, `RobustScaler` y sin escalado; compara.
3. Calcula estabilidad de clusters con varias semillas/muestras bootstrap.
4. Compara PCA, t-SNE y UMAP en `digits`.
5. Aplica GMM y usa sus probabilidades para detectar observaciones ambiguas.
6. Investiga HDBSCAN y spectral clustering.
